In [ ]:
#setup
import io
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.options.display.precision = 3
pd.options.display.max_columns = 30
pd.options.display.width = 140
rng = np.random.default_rng(42)

PROJECT = Path.cwd().parent                       # the notebook lives in notebooks/, so the project is one level up
DATA = PROJECT / "data"
DATA.mkdir(exist_ok=True)
print(sys.executable)
print("pandas", pd.__version__)
print("project folder:", PROJECT, "| looks right:", (PROJECT / "requirements.txt").exists())

In [ ]:
#the generator, and the CSV
PRODUCTS = {                                      # product: (category, unit price)
    "Notebook": ("Stationery", 45.5), "Pen": ("Stationery", 8.25), "Marker": ("Stationery", 32.0),
    "Bag": ("Accessories", 1299.0), "Bottle": ("Accessories", 349.0),
    "Headphones": ("Electronics", 1999.0), "Mouse": ("Electronics", 599.0),
}

def make_sales(n=1500, seed=42):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "order_date": pd.Timestamp("2026-07-01") + pd.to_timedelta(r.integers(0, 120, n), unit="D"),
        "city": r.choice(["Delhi", "Mumbai", "Pune", "Chennai", "Kolkata"], n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
        "product": r.choice(list(PRODUCTS), n, p=[0.22, 0.25, 0.12, 0.12, 0.10, 0.07, 0.12]),
        "channel": r.choice(["web", "app", "store"], n, p=[0.45, 0.35, 0.20]),
        "qty": r.poisson(2.2, n) + 1,
        "discount": r.choice([0.0, 0.05, 0.10, 0.20], n, p=[0.5, 0.2, 0.2, 0.1]),
        "customer_age": np.clip(r.normal(34, 11, n), 18, 75).round(),
        "rating": r.choice([1.0, 2.0, 3.0, 4.0, 5.0], n, p=[0.05, 0.08, 0.17, 0.35, 0.35]),
    })
    df = df.sort_values("order_date").reset_index(drop=True)
    df.insert(0, "order_id", np.arange(5001, 5001 + n))
    df["category"] = df["product"].map(lambda p: PRODUCTS[p][0])
    df["unit_price"] = df["product"].map(lambda p: PRODUCTS[p][1])
    p_return = np.where(df["category"] == "Electronics", 0.12, 0.04)       # electronics come back more often
    df["returned"] = r.random(n) < p_return
    df["revenue"] = (df["qty"] * df["unit_price"] * (1 - df["discount"])).round(2)

    # ----- inject realistic mess -----
    df.loc[r.choice(n, 75, replace=False), "customer_age"] = np.nan        # missing ages
    df.loc[r.choice(n, 180, replace=False), "rating"] = np.nan             # many customers never rate
    df.loc[[10, 400], "customer_age"] = [190, 3]                           # impossible ages
    extreme = [25, 700, 1100]
    df.loc[extreme, "qty"] = [90, 120, 75]                                 # extreme quantities
    df.loc[extreme, "revenue"] = (df.loc[extreme, "qty"] * df.loc[extreme, "unit_price"]).round(2)
    df.loc[r.choice(n, 20, replace=False), "city"] = np.nan                # missing cities
    df.loc[r.choice(n, 15, replace=False), "city"] = "delhi"               # inconsistent capitalisation
    df.loc[r.choice(n, 10, replace=False), "city"] = "Pune "               # a trailing space
    return pd.concat([df, df.sample(6, random_state=3)], ignore_index=True)    # 6 duplicated rows

make_sales().to_csv(DATA / "sales_messy.csv", index=False)
print("saved", DATA / "sales_messy.csv")

In [ ]:
#load it as you would a real file
sales = pd.read_csv(DATA / "sales_messy.csv", parse_dates=["order_date"])
print(sales.shape)
sales.info()
sales.head()

In [ ]:
#the quality checklist
print(sales.tail(3), "\n")
print("a random sample:\n", sales.sample(4, random_state=1), "\n")

missing = sales.isna().sum()
print("missing values per column:\n", missing[missing > 0], "\n")
print("share missing (%):\n", (sales.isna().mean() * 100).round(1)[lambda s: s > 0], "\n")

print("fully duplicated rows:", sales.duplicated().sum(), "| repeated order_ids:", sales["order_id"].duplicated().sum())
print(sales[sales.duplicated(keep=False)].sort_values("order_id").head(4), "\n")
print("distinct values per column:\n", sales.nunique())

In [ ]:
#text columns need value_counts
print(sales["city"].value_counts(dropna=False), "\n")       # dropna=False shows the missing group too
print(sales["product"].value_counts(normalize=True).round(3), "\n")
print(sales["channel"].value_counts().to_dict())

In [ ]:
#impossible and suspicious values
age_ok = sales["customer_age"].between(18, 75)
print("impossible ages:\n", sales.loc[~age_ok & sales["customer_age"].notna(), ["order_id", "customer_age"]], "\n")
print("largest quantities:\n", sales.nlargest(5, "qty")[["order_id", "product", "qty", "revenue"]], "\n")
print("dates run from", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
print("negative prices or revenue:", (sales["unit_price"] <= 0).sum(), (sales["revenue"] < 0).sum())

In [ ]:
#minimal fixes so the rest of the EDA works
print("city values before:", sorted(sales["city"].dropna().unique()))
sales["city"] = sales["city"].str.strip().str.title()
sales = sales.drop_duplicates().reset_index(drop=True)
print("city values after :", sorted(sales["city"].dropna().unique()))
print("rows now:", len(sales), "(the 6 duplicates are gone)")

In [ ]:
#the numeric summary
num_cols = ["qty", "discount", "customer_age", "rating", "unit_price", "revenue"]
print(sales[num_cols].describe().T)                         # one row per column is easier to read

In [ ]:
#more percentiles, text, and dates
print(sales[num_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T, "\n")

text_cols = ["city", "product", "category", "channel"]
print(sales[text_cols].describe().T, "\n")                  # count, unique, top, freq
print(sales["order_date"].describe(), "\n")
print(sales[["qty", "revenue"]].agg(["mean", "median", "std", "skew", "min", "max"]).T)
print("\nskewness of every numeric column:\n", sales[num_cols].skew().round(2))

In [ ]:
#the IQR rule from S2, applied to columns
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ["qty", "customer_age", "revenue", "rating"]:
    low, high = iqr_bounds(sales[col])
    flagged = sales[(sales[col] < low) | (sales[col] > high)]
    print(f"{col:<13} fences ({low:9.1f}, {high:9.1f}) -> {len(flagged):>3} flagged")

In [ ]:
#look at the shapes
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col in zip(axes, ["qty", "customer_age", "revenue"]):
    sales[col].plot.hist(bins=40, ax=ax, color="lightsteelblue", edgecolor="white")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
#look at the shapes
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col in zip(axes, ["qty", "customer_age", "revenue"]):
    sales[col].plot.hist(bins=40, ax=ax, color="lightsteelblue", edgecolor="white")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
#Boolean slicing and query
#masks
mask = sales["city"] == "Delhi"
print(mask.dtype, "| True count:", mask.sum(), "| proportion:", round(mask.mean(), 3))
print(sales[mask].shape)
print(sales.loc[mask, ["order_id", "product", "revenue"]].head(3))

In [ ]:
#combining conditions
big = sales["revenue"] > 5000
web = sales["channel"] == "web"
print("big and web   :", (big & web).sum())
print("big or web    :", (big | web).sum())
print("big, not web  :", (big & ~web).sum())

print("qty between 2 and 4  :", sales["qty"].between(2, 4).sum())
print("Pen or Marker        :", sales["product"].isin(["Pen", "Marker"]).sum())
print("name contains 'book' :", sales["product"].str.contains("book", case=False).sum())
print("starts with M        :", sales["product"].str.startswith("M").sum())
print("rating missing       :", sales["rating"].isna().sum(), "| has a rating:", sales["rating"].notna().sum())

recent = sales["order_date"] >= "2026-10-01"                 # a string is compared as a date
print("orders since 1 October:", recent.sum())
print(sales.loc[big & web & recent, ["order_id", "product", "qty", "revenue"]].head())

In [ ]:
#the traps with missing values
n_nan_city = sales["city"].isna().sum()
print(f"city != 'Delhi' counts {(sales['city'] != 'Delhi').sum()} rows, and that includes the {n_nan_city} rows with NO city")

print("rating < 3 :", (sales["rating"] < 3).sum())
print("rating >= 3:", (sales["rating"] >= 3).sum())
print("missing    :", sales["rating"].isna().sum())
print("the three add up to the row count:",
      (sales["rating"] < 3).sum() + (sales["rating"] >= 3).sum() + sales["rating"].isna().sum() == len(sales))

hits = sales["city"].str.contains("del", case=False, na=False)    # na=False states what a missing city should count as
print("cities containing 'del':", hits.sum())

In [ ]:
#query, SQL-like filtering
print(sales.query("revenue > 5000 and channel == 'web'").shape[0])
print(sales.query("product in ['Pen', 'Marker'] and qty >= 5").shape[0])
print(sales.query("returned").shape[0], sales.query("not returned").shape[0])
print(sales.query("20 <= customer_age <= 30").shape[0])

min_rev, wanted = 3000, ["Delhi", "Pune"]
print(sales.query("revenue >= @min_rev and city in @wanted").shape[0])      # @ brings in a Python variable
print(sales.query("customer_age.isna()", engine="python").shape[0])           # methods need engine="python"

spaced = pd.DataFrame({"unit price": [10, 20, 30], "qty": [1, 2, 3]})
print(spaced.query("`unit price` > 15"))                                      # backticks for awkward column names

print("same rows as the mask version:", sales.query("revenue > 5000").equals(sales[sales["revenue"] > 5000]))

In [ ]:
#the errors
try:
    sales[(sales["revenue"] > 5000) and (sales["channel"] == "web")]
except ValueError as e:
    print("ValueError:", e)

try:
    sales[sales["revenue"] > 5000 & sales["channel"] == "web"]          # & binds tighter than >, so this is wrong
except Exception as e:
    print(type(e).__name__ + ":", str(e)[:90])

delhi = sales[sales["city"] == "Delhi"].copy()                            # .copy() if you plan to change the result
delhi["flag"] = delhi["revenue"] > 1000
print(delhi.shape, "| the original gained a 'flag' column:", "flag" in sales.columns)

In [ ]:
#groupby
#the basics
print(sales.groupby("city")["revenue"].sum().sort_values(ascending=False).round(0), "\n")
print(sales.groupby("city")["revenue"].agg(["count", "sum", "mean", "median"]).round(1), "\n")
print(sales.groupby("channel")[["qty", "revenue"]].mean().round(2))

print("\nrows in the city groups:", sales.groupby("city").size().sum(), "of", len(sales),
      "| with dropna=False:", sales.groupby("city", dropna=False).size().sum())

In [ ]:
#named aggregation, the readable way
summary = sales.groupby("city").agg(
    orders=("order_id", "count"),
    units=("qty", "sum"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    median_order=("revenue", "median"),
    p90_order=("revenue", lambda s: s.quantile(0.9)),
    return_rate=("returned", "mean"),                      # the mean of True/False is the proportion
    avg_rating=("rating", "mean"),
)
print(summary.round(2).sort_values("revenue", ascending=False))

In [ ]:
#several keys, unstack, and as_index
by_two = sales.groupby(["category", "channel"])["revenue"].sum()
print(by_two.round(0), "\n")                                  # a Series with a two-level index
print(by_two.unstack().round(0), "\n")                         # the second key becomes columns
print(sales.groupby(["category", "channel"], as_index=False)["revenue"].sum().head(4))   # keys stay ordinary columns
print("\nlook up one group:", round(by_two.loc[("Electronics", "web")], 0))

In [ ]:
#count vs size
g = sales.groupby("channel")
print(pd.DataFrame({"size (all rows)": g.size(),
                    "count (non-missing rating)": g["rating"].count(),
                    "missing rating": g["rating"].apply(lambda s: s.isna().sum())}))

In [ ]:
#transform, results that keep the original rows
work = sales.copy()
work["city_total"] = work.groupby("city")["revenue"].transform("sum")
work["share_of_city"] = (work["revenue"] / work["city_total"] * 100).round(3)
work["revenue_z"] = work.groupby("product")["revenue"].transform(lambda s: (s - s.mean()) / s.std())
print(work[["order_id", "city", "product", "revenue", "share_of_city", "revenue_z"]].head(5))
print("rows:", len(work), "| group means of revenue_z (should be about 0):",
      work.groupby("product")["revenue_z"].mean().round(6).tolist())

work["rank_in_city"] = work.groupby("city")["revenue"].rank(ascending=False, method="first")
print("\nthe biggest order in each city:\n",
      work[work["rank_in_city"] == 1][["city", "order_id", "product", "qty", "revenue"]].sort_values("city"))

In [ ]:
#filter, and top-n per group
common = sales.groupby("product").filter(lambda g: len(g) > 120)       # keep whole groups that pass a test
print("products kept:", sorted(common["product"].unique()), "| dropped:", sorted(set(sales["product"]) - set(common["product"])))

top2 = sales.sort_values("revenue", ascending=False).groupby("city").head(2)
print(top2.sort_values(["city", "revenue"], ascending=[True, False])[["city", "order_id", "product", "revenue"]])

In [ ]:
#time, month and weekday
sales["month"] = sales["order_date"].dt.to_period("M")
monthly = sales.groupby("month").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
                                     active_days=("order_date", lambda s: s.dt.normalize().nunique()))
monthly["revenue_per_day"] = (monthly["revenue"] / monthly["active_days"]).round(0)
print(monthly, "\n")

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = sales.groupby(sales["order_date"].dt.day_name())["revenue"].agg(["count", "sum"]).reindex(order)
print(by_day.round(0))

In [ ]:
#the average-of-averages trap again (S1)
by_city = sales.groupby("city")["revenue"]
print("mean of the city means:", round(by_city.mean().mean(), 2))
print("overall mean (orders with a city):", round(sales.loc[sales["city"].notna(), "revenue"].mean(), 2))
print("weighted mean of the city means:", round(np.average(by_city.mean(), weights=by_city.size()), 2))

In [ ]:
#crosstab and pivot_table
#counts and proportions
print(pd.crosstab(sales["category"], sales["channel"]), "\n")
print(pd.crosstab(sales["category"], sales["channel"], margins=True, margins_name="Total"), "\n")
print("each ROW sums to 1:\n", pd.crosstab(sales["category"], sales["channel"], normalize="index").round(3), "\n")
print("each COLUMN sums to 1:\n", pd.crosstab(sales["category"], sales["channel"], normalize="columns").round(3), "\n")
print("all cells sum to 1:\n", pd.crosstab(sales["category"], sales["channel"], normalize=True).round(3))

In [ ]:
#values with an aggregation, and a link back to S7
print(pd.crosstab(sales["city"], sales["category"], values=sales["revenue"], aggfunc="sum").round(0), "\n")

returns = pd.crosstab(sales["category"], sales["returned"])
print(returns, "\n")
print("return rate by category (%):\n", (pd.crosstab(sales["category"], sales["returned"], normalize="index")[True] * 100).round(1), "\n")

chi2, p, dof, expected = stats.chi2_contingency(returns)               # S7: are category and returns related?
print(f"chi-square = {chi2:.2f}, df = {dof}, p = {p:.2e}, minimum expected count = {expected.min():.1f}")

In [ ]:
#pivot_table
pt = sales.pivot_table(index="city", columns="category", values="revenue", aggfunc="sum",
                       margins=True, margins_name="Total", fill_value=0)
print(pt.round(0), "\n")

print(sales.pivot_table(index="category", values=["revenue", "qty"], aggfunc={"revenue": "sum", "qty": "mean"}).round(2), "\n")
print(sales.pivot_table(index="channel", columns="returned", values="order_id", aggfunc="count"))

In [ ]:
#Two numbers, and binning
#after excluding the impossible values
usable = sales[sales["customer_age"].between(18, 75) & (sales["qty"] <= 20)]
print(len(usable), "of", len(sales), "rows kept\n")
print(usable[num_cols].corr().round(2), "\n")                          # Pearson by default
print(usable[num_cols].corr(method="spearman").round(2))

In [ ]:
#cut turns numbers into groups
bands = pd.cut(usable["customer_age"], bins=[17, 25, 35, 45, 60, 75],
               labels=["18-25", "26-35", "36-45", "46-60", "61-75"])
print(usable.groupby(bands, observed=True).agg(orders=("order_id", "count"), avg_revenue=("revenue", "mean"),
                                               avg_rating=("rating", "mean"), return_rate=("returned", "mean")).round(3))

In [ ]:
#the analysis
print("1. rows", len(sales), "| columns", sales.shape[1], "| period", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
print("   missing:", {k: int(v) for k, v in sales.isna().sum().items() if v})

print("\n2. revenue by city and category")
print(sales.groupby("city")["revenue"].sum().sort_values(ascending=False).round(0).to_dict())
print(sales.groupby("category")["revenue"].sum().sort_values(ascending=False).round(0).to_dict())
print("   skew of revenue:", round(sales["revenue"].skew(), 2), "| mean", round(sales["revenue"].mean()), "vs median", round(sales["revenue"].median()))

print("\n3. channels")
print(sales.groupby("channel").agg(orders=("order_id", "count"), avg_order=("revenue", "mean"), return_rate=("returned", "mean")).round(3))

print("\n4. returns by category")
rate = sales.groupby("category")["returned"].mean().round(3)
chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(sales["category"], sales["returned"]))
print(rate.to_dict(), f"| chi-square p = {p:.2e}")

print("\n5. revenue per active day")
print(monthly["revenue_per_day"].to_dict())

print("\n6. rating by channel")
groups = [g["rating"].dropna().to_numpy() for _, g in sales.groupby("channel")]
print(sales.groupby("channel")["rating"].agg(["count", "mean"]).round(3).to_dict("index"))
print("   ANOVA:", stats.f_oneway(*groups))

print("\n7. suspect records")
print("   impossible ages :", int((~sales["customer_age"].between(18, 75) & sales["customer_age"].notna()).sum()))
print("   extreme quantity:", int((sales["qty"] > 20).sum()))
print("   missing city    :", int(sales["city"].isna().sum()), "| missing rating:", int(sales["rating"].isna().sum()))